In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['hatch.linewidth'] = 0.2
import numpy as np
import polars as pl

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.file_locations import intermediate_files_location
from src.plot_helpers import make_histogram_plot
from src.ntuple_variables.variables import combined_training_vars
from src.df_helpers import lazy_height

# Data Loading

In [ ]:
# Load the full dataset (lazy)
print("Loading all_df.parquet...")
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")
print(f"Total events: {lazy_height(all_df)}")

# Filter: remove overlay files, require valid Enu, exclude run 4a
presel_df = (
    all_df
    .filter(~pl.col("filetype").is_in([
        "isotropic_one_gamma_overlay", "delete_one_gamma_overlay",
        "numuCC_rad_corrected", "NC_coherent_1g_reweighted"
    ]))
    .filter(pl.col("wc_kine_reco_Enu") > 0)
    .filter(pl.col("normalizing_run_period") != "4a")
)

# --- Selection Definitions ---
# Define as Polars expressions so they can be reused on any dataframe (main df and detvar df).

# Erin inclusive 1g selection
erin_1g_sel_expr = (
    (pl.col("wc_shw_sp_n_20mev_showers") > 0) &
    (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0) &
    (pl.col("wc_single_photon_numu_score") > 0.4) &
    (pl.col("wc_single_photon_other_score") > 0.2) &
    (pl.col("wc_single_photon_ncpi0_score") > -0.05) &
    (pl.col("wc_single_photon_nue_score") > -1.0) &
    (pl.col("wc_shw_sp_n_20br1_showers") == 1)
)

# NC pi0 sideband: relaxed numu/other scores, inverted ncpi0 score
erin_ncpi0_sel_expr = (
    (pl.col("wc_shw_sp_n_20mev_showers") > 0) &
    (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0) &
    (pl.col("wc_single_photon_numu_score") > 0.1) &
    (pl.col("wc_single_photon_other_score") > -0.4) &
    (pl.col("wc_single_photon_ncpi0_score") < -0.4) &
    (pl.col("wc_single_photon_nue_score") > -20.0)
)

# Add selection flags
presel_df = presel_df.with_columns([
    erin_1g_sel_expr.cast(pl.Int32).alias("erin_inclusive_1g_sel"),
    erin_ncpi0_sel_expr.cast(pl.Int32).alias("erin_nc_pi0_sideband_flag"),
])

In [ ]:
# Select physics variables (drop training-only variables to save memory)
load_vars = [col for col in presel_df.collect_schema().names() if col not in combined_training_vars]

extra_vars = [
    "wc_kine_reco_Enu",
    "wc_reco_num_protons_35_MeV",
    "wc_reco_backwards_projected_dist",
    "wc_reco_distance_to_boundary",
    "wc_reco_shower_theta",
    "wc_reco_shower_phi",
    "wc_kine_pio_mass",
    "lantern_diphoton_mass",

    "wc_reco_num_protons_35_MeV_75cm_from_wcvtx",
    "wc_reco_num_protons_35_MeV_75cm_from_wcshwrvtx",
    "wc_reco_num_protons_35_MeV_75cm_from_wc2shwvtx",
    "wc_reco_num_protons_35_MeV_75cm_from_lanternvtx",
    "wc_reco_num_protons_35_MeV_75cm_from_gleevtx",
    "wc_reco_num_protons_35_MeV_75cm_from_pandoravtx",
    "wc_blip_nWithin_75cm",
    "pandora_pelee_blip_nWithin_75cm",
    "pandora_glee_blip_nWithin_75cm",
    "lantern_blip_nWithin_75cm",
    "wc_reco_num_protons_35_MeV_75cm_from_wcvtx_plus_backtrack_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_wcvtx_plus_wcvtx_1cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_wcvtx_plus_wcvtx_3cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_wcvtx_plus_wcvtx_10cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_wcshwrvtx_plus_backtrack_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_wcshwrvtx_plus_wcshwrvtx_1cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_wcshwrvtx_plus_wcshwrvtx_3cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_wcshwrvtx_plus_wcshwrvtx_10cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_wc2shwvtx_plus_backtrack_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_wc2shwvtx_plus_wc2shwvtx_1cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_wc2shwvtx_plus_wc2shwvtx_3cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_wc2shwvtx_plus_wc2shwvtx_10cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_lanternvtx_plus_backtrack_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_lanternvtx_plus_lanternvtx_1cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_lanternvtx_plus_lanternvtx_3cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_lanternvtx_plus_lanternvtx_10cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_gleevtx_plus_backtrack_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_gleevtx_plus_gleevtx_1cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_gleevtx_plus_gleevtx_3cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_gleevtx_plus_gleevtx_10cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_pandoravtx_plus_backtrack_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_pandoravtx_plus_pandoravtx_1cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_pandoravtx_plus_pandoravtx_3cm_vertex_blips",
    "wc_reco_num_protons_35_MeV_75cm_from_pandoravtx_plus_pandoravtx_10cm_vertex_blips",
]
for var in extra_vars:
    if var not in load_vars:
        load_vars.append(var)

presel_merged_df = presel_df.select(load_vars).collect()
print(presel_merged_df.shape)

# Plot Helpers

Common parameters are collected into dicts so each `make_histogram_plot` call only specifies
what's unique to that plot.

In [ ]:
# Blip-based neutron (Nn/0n) split expressions
oneshwzero_n_expr = (
    (pl.col("blip_no_shower_cone_no_backtrack_cones_n") <= 2) &
    (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") <= 11)
)
oneshwNn_expr = (
    (pl.col("blip_no_shower_cone_no_backtrack_cones_n") > 2) |
    (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") > 11)
)

twoshwzero_n_expr = (
    (pl.col("blip_2shwr_wcvtx_no_shower_cones_n") + pl.col("blip_2shwr_wcvtx_nWithin_3cm") <= 2) &
    (pl.col("blip_2shwr_wcvtx_no_shower_cones_sumE") + pl.col("blip_2shwr_wcvtx_sumE_3cm") <= 11)
)
twoshwNn_expr = (
    (pl.col("blip_2shwr_wcvtx_no_shower_cones_n") + pl.col("blip_2shwr_wcvtx_nWithin_3cm") > 2) |
    (pl.col("blip_2shwr_wcvtx_no_shower_cones_sumE") + pl.col("blip_2shwr_wcvtx_sumE_3cm") > 11)
)

# Proton count variables reused across many plots
proton_bins = np.linspace(0, 4, 5)
proton_count_var = "wc_reco_num_protons_35_MeV"
proton_count_display_var = "WC Reconstructed num protons (35 MeV threshold)"

oneshw_proton_blip_var = "wc_reco_num_protons_35_MeV_plus_backtrack_blips"
oneshwproton_blip_display_var = "WC Reconstructed num protons (35 MeV) + backtrack blips"

twoshw_proton_blip_var = "wc_reco_num_protons_35_MeV_plus_3cm_vertex_blips"
twoshwproton_blip_display_var = "WC Reconstructed num protons (35 MeV) + 3cm vertex blips"


# NC Pi0 Sideband

In [ ]:
erin_ncpi0_sel_df = presel_merged_df.filter(pl.col("erin_nc_pi0_sideband_flag") == 1)
erin_ncpi0_sel_crt_df = erin_ncpi0_sel_df.filter(pl.col("pandora_crtveto") == 0)
erin_ncpi0_sel_crt_0p_df = erin_ncpi0_sel_crt_df.filter(pl.col("wc_reco_num_protons_35_MeV") == 0)
erin_ncpi0_sel_crt_1shwNn_df = erin_ncpi0_sel_crt_df.filter(oneshwNn_expr)
erin_ncpi0_sel_crt_1shw0n_df = erin_ncpi0_sel_crt_df.filter(oneshwzero_n_expr)
erin_ncpi0_sel_crt_2shwNn_df = erin_ncpi0_sel_crt_df.filter(twoshwNn_expr)
erin_ncpi0_sel_crt_2shw0n_df = erin_ncpi0_sel_crt_df.filter(twoshwzero_n_expr)


In [ ]:
for blip_var in [
    "wc_truth_dist",
    "pandora_truth_dist",
    "lantern_truth_dist",
    "glee_truth_dist",
]:
    make_histogram_plot(
        pred_and_data_sel_df=erin_ncpi0_sel_crt_0p_df,
        bins=np.linspace(0, 50, 26),
        var=blip_var,
        display_var=blip_var,
        title="Erin NC Pi0 Sideband with CRT veto and WC 0p cut",
        selname="erin_ncpi0_0p_sel_crt",
        breakdown_type="erin_Np0pNn0n_pi0_categories",
    )

In [ ]:
for blip_var in [
    "blip_2shwr_min_wcvtx_to_shower_vtx_dist",
    "blip_2shwr_min_wc2shwvtx_to_shower_vtx_dist",
    "blip_2shwr_min_lanternvtx_to_shower_vtx_dist",
    "blip_2shwr_min_gleevtx_to_shower_vtx_dist",
    "blip_2shwr_min_pandoravtx_to_shower_vtx_dist",
]:
    make_histogram_plot(
        pred_and_data_sel_df=erin_ncpi0_sel_crt_0p_df,
        bins=np.linspace(0, 50, 26),
        var=blip_var,
        display_var=blip_var,
        title="Erin NC Pi0 Sideband with CRT veto and WC 0p cut",
        selname="erin_ncpi0_0p_sel_crt",
        breakdown_type="erin_Np0pNn0n_pi0_categories",
    )

# 2D Scatter: Reco-True Distance vs Reco-Min-Shower Distance

In [ ]:
reconstructions = [
    {
        "label": "WC vtx",
        "truth_dist_var": "wc_truth_dist",
        "min_shw_dist_var": "blip_2shwr_min_wcvtx_to_shower_vtx_dist",
    },
    {
        "label": "WC 2shw vtx",
        "truth_dist_var": "wc_truth_dist",
        "min_shw_dist_var": "blip_2shwr_min_wc2shwvtx_to_shower_vtx_dist",
    },
    {
        "label": "Pandora",
        "truth_dist_var": "pandora_truth_dist",
        "min_shw_dist_var": "blip_2shwr_min_pandoravtx_to_shower_vtx_dist",
    },
    {
        "label": "Lantern",
        "truth_dist_var": "lantern_truth_dist",
        "min_shw_dist_var": "blip_2shwr_min_lanternvtx_to_shower_vtx_dist",
    },
    {
        "label": "GLEE",
        "truth_dist_var": "glee_truth_dist",
        "min_shw_dist_var": "blip_2shwr_min_gleevtx_to_shower_vtx_dist",
    },
]

true_nc_1pi0_0p_expr = (
    pl.col("normal_overlay") &
    pl.col("wc_truth_inFV") &
    pl.col("wc_truth_isNC") &
    ~pl.col("wc_truth_NCDeltaRad") &
    pl.col("wc_truth_1pi0") &
    pl.col("wc_truth_0p")
)

plot_df = erin_ncpi0_sel_crt_0p_df.filter(true_nc_1pi0_0p_expr)

ncols = 3
nrows = (len(reconstructions) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = axes.flatten()

for i, reco in enumerate(reconstructions):
    ax = axes[i]
    x_var = reco["truth_dist_var"]
    y_var = reco["min_shw_dist_var"]

    sub = plot_df.select([x_var, y_var]).drop_nulls()
    x = sub[x_var].to_numpy()
    y = sub[y_var].to_numpy()

    ax.scatter(x, y, s=2, alpha=0.3, rasterized=True)
    ax.set_xlabel(x_var, fontsize=9)
    ax.set_ylabel(y_var, fontsize=9)
    ax.set_title(reco["label"])
    ax.set_xlim(0, 50)
    ax.set_ylim(0, 50)
    ax.axline((0, 0), slope=1, color="red", linestyle="--", linewidth=0.8, label="y=x")
    ax.legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("NC Pi0 Sideband, CRT veto, WC 0p: Reco-True Dist vs Reco-Min-Shower Dist\n(true NC $1\\pi^0$ $0p$ events only)", fontsize=12)
plt.tight_layout()
plt.show()